# LearnIS — AI-Powered Cognitive Tutor
> **Guiding learners through questions, not answers.**

**Gemma 4 Good Hackathon 2026 | Kaggle x Google DeepMind**

---

## What is LearnIS?

LearnIS (Learning Intelligence System) is a Socratic tutoring system powered by **Gemma 4**. Instead of giving answers, it guides learners step by step through their own reasoning process.

Grounded in three pedagogical frameworks:
- **Socratic Method** — knowledge through guided questioning
- **Constructivism** (Vygotsky, Piaget) — building on prior knowledge
- **Cognitive Scaffolding** — adaptive support that fades as autonomy grows

### Key Metrics tracked in real time:
| Metric | Description |
|---|---|
| **Mastery Score** | How well the learner understands the concept (0-100) |
| **AI Dependency Indicator (ADI)** | How much the learner relies on AI help (0-100, lower is better) |
| **Hint Level** | Current scaffolding intensity (0=none, 3=maximum) |

---

## Step 1 — Install Dependencies

In [ ]:
!pip install -q keras-hub keras

## Step 2 — Import Libraries

In [ ]:
import os
import json

os.environ['KERAS_BACKEND'] = 'jax'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'

import keras
import keras_hub

print('Keras version:', keras.__version__)
print('Setup complete.')

## Step 3 — Load Gemma 4
We use **Gemma 4 31B instruct** — the most capable model available on Kaggle for following complex pedagogical instructions.

In [ ]:
gemma = keras_hub.models.GemmaCausalLM.from_preset(
    'kaggle://keras/gemma4/keras/gemma4_instruct_31b'
)
print('Gemma 4 31B loaded successfully.')

## Step 4 — LearnIS Pedagogical System Prompt

This is the technical core of LearnIS. The system prompt encodes:
- The Socratic method (never give direct answers)
- Cognitive scaffolding rules (adaptive hint levels 0 to 3)
- Real-time metrics via structured JSON output (mastery score, AI Dependency Indicator)
- Conceptual trap injection to detect misconceptions

In [ ]:
LEARNIS_SYSTEM_PROMPT = """
You are LearnIS, an AI cognitive tutor built on the Socratic method.

YOUR CORE ROLE:
You are NOT an answer machine. You are a thinking partner.
Your mission is to help the learner reach understanding through their own reasoning effort.
You NEVER provide a complete, direct answer to the learner question.

PEDAGOGICAL PROTOCOL (follow strictly at every turn):
1. EXPLORATION: If first turn, probe prior knowledge with one open question.
2. GUIDED QUESTIONING: Ask one precise question that moves the learner one step closer to understanding.
3. ORIENTATION: If the learner is stuck, offer a partial analogy or example. Never reveal the full answer.
4. CONCEPTUAL TRAPS: Occasionally introduce subtle misconceptions to test whether the learner can identify them.
5. ASSESSMENT: After each learner response, update your evaluation of mastery and autonomy.

SCAFFOLDING RULES:
- hint_level 0: Pure Socratic question. No hint at all.
- hint_level 1: Guiding analogy or example without revealing the answer.
- hint_level 2: Partial framework or structure to help organize thinking.
- hint_level 3: Strong conceptual pointer. Still no direct answer.
Increase hint_level when learner is clearly blocked or repeating errors.
Decrease hint_level when learner shows autonomous reasoning.

AI DEPENDENCY INDICATOR (ADI):
Start at 50. Range 0 to 100.
Decrease when learner reasons independently and builds on prior answers.
Increase when learner asks for direct answers or shows no reasoning effort.

MASTERY SCORE:
Start at 0. Range 0 to 100.
Increase progressively as the learner demonstrates genuine understanding.

OUTPUT FORMAT (mandatory, always return valid JSON only, no text outside the JSON):
{
  "tutor_message": "Your Socratic response to the learner in natural language",
  "hint_level": 0,
  "mastery_score": 0,
  "ai_dependency_indicator": 50,
  "internal_note": "Brief note on learner reasoning quality not shown to learner"
}
"""

print('System prompt configured.')
print(f'Prompt length: {len(LEARNIS_SYSTEM_PROMPT)} characters')

## Step 5 — LearnIS Dialogue Engine

In [ ]:
class LearnIS:
    """
    LearnIS Cognitive Tutor powered by Gemma 4.
    Implements Socratic dialogue with real-time cognitive metrics:
    - Mastery Score: tracks concept understanding (0-100)
    - AI Dependency Indicator (ADI): tracks cognitive autonomy (0-100, lower is better)
    - Hint Level: current scaffolding intensity (0-3)
    """

    def __init__(self, model, subject):
        self.model = model
        self.subject = subject
        self.history = []
        self.mastery_score = 0
        self.adi = 50
        self.hint_level = 0
        self.turn = 0

    def _build_prompt(self, learner_input):
        """Build the full conversation prompt with system instructions and history."""
        prompt = f'<start_of_turn>system\n{LEARNIS_SYSTEM_PROMPT}<end_of_turn>\n'
        prompt += f'<start_of_turn>user\nSubject to learn: {self.subject}<end_of_turn>\n'
        for t in self.history:
            prompt += f'<start_of_turn>user\n{t["learner"]}<end_of_turn>\n'
            prompt += f'<start_of_turn>model\n{json.dumps(t["tutor"])}<end_of_turn>\n'
        prompt += f'<start_of_turn>user\n{learner_input}<end_of_turn>\n'
        prompt += '<start_of_turn>model\n'
        return prompt

    def _parse_response(self, raw):
        """Extract and parse JSON from model output."""
        try:
            start = raw.find('{')
            end = raw.rfind('}') + 1
            return json.loads(raw[start:end])
        except Exception:
            return {
                'tutor_message': raw.strip(),
                'hint_level': self.hint_level,
                'mastery_score': self.mastery_score,
                'ai_dependency_indicator': self.adi,
                'internal_note': 'JSON parsing failed, raw response used.'
            }

    def _display_metrics(self, r):
        """Display real-time pedagogical metrics after each turn."""
        mastery = r['mastery_score']
        adi = r['ai_dependency_indicator']
        hint = r['hint_level']
        mb = chr(9608) * (mastery // 10) + chr(9617) * (10 - mastery // 10)
        ab = chr(9608) * (adi // 10) + chr(9617) * (10 - adi // 10)
        print('\n' + '=' * 60)
        print(' LEARNIS COGNITIVE METRICS')
        print('=' * 60)
        print(f' Mastery Score       [{mb}] {mastery}/100')
        print(f' AI Dependency (ADI) [{ab}] {adi}/100  (lower = more autonomous)')
        print(f' Scaffolding Level   {hint}/3')
        print(f' Session Turn        {self.turn}')
        print('=' * 60 + '\n')

    def respond(self, learner_input):
        """Process learner input and return a Socratic tutor response."""
        self.turn += 1
        prompt = self._build_prompt(learner_input)
        raw = self.model.generate(prompt, max_length=512)
        response = self._parse_response(raw)
        self.mastery_score = response.get('mastery_score', self.mastery_score)
        self.adi = response.get('ai_dependency_indicator', self.adi)
        self.hint_level = response.get('hint_level', self.hint_level)
        self.history.append({'learner': learner_input, 'tutor': response})
        print(f'\nLearnIS: {response["tutor_message"]}')
        self._display_metrics(response)
        return response

print('LearnIS engine ready.')

## Step 6 — Live Demos

Three Socratic sessions on different subjects. Each demo shows how LearnIS guides the learner without giving direct answers, while tracking cognitive metrics in real time.

### Demo A — Mathematics: The Pythagorean Theorem

In [ ]:
print('LEARNIS DEMO A — Mathematics: The Pythagorean Theorem')
print('=' * 60)

tutor_a = LearnIS(gemma, subject='The Pythagorean Theorem')

print('Learner: I know the Pythagorean theorem has something to do with triangles.')
tutor_a.respond('I know the Pythagorean theorem has something to do with triangles.')

print('Learner: I think it says a squared plus b squared equals c squared?')
tutor_a.respond('I think it says a squared plus b squared equals c squared?')

print('Learner: But what exactly is c? Is it any side of the triangle?')
tutor_a.respond('But what exactly is c? Is it any side of the triangle?')

### Demo B — Philosophy: Critical Thinking

In [ ]:
print('LEARNIS DEMO B — Philosophy: Critical Thinking')
print('=' * 60)

tutor_b = LearnIS(gemma, subject='Critical thinking and its importance')

print('Learner: Critical thinking means thinking a lot before making decisions.')
tutor_b.respond('Critical thinking means thinking a lot before making decisions.')

print('Learner: So it is basically just being careful and not rushing?')
tutor_b.respond('So it is basically just being careful and not rushing?')

print('Learner: I guess it also involves questioning what people tell you?')
tutor_b.respond('I guess it also involves questioning what people tell you?')

### Demo C — Science: Photosynthesis

In [ ]:
print('LEARNIS DEMO C — Science: Photosynthesis')
print('=' * 60)

tutor_c = LearnIS(gemma, subject='Photosynthesis in plants')

print('Learner: Photosynthesis is how plants make their food using sunlight.')
tutor_c.respond('Photosynthesis is how plants make their food using sunlight.')

print('Learner: They absorb sunlight through their leaves I think.')
tutor_c.respond('They absorb sunlight through their leaves I think.')

print('Learner: And they release oxygen as a result?')
tutor_c.respond('And they release oxygen as a result?')

## Step 7 — Interactive Mode

Enter your own topic and have a real Socratic session with LearnIS. Type `quit` to end the session and see your summary.

In [ ]:
subject = input('Enter the topic you want to learn about: ')
session = LearnIS(gemma, subject=subject)

print(f'\nLearnIS session started on: {subject}')
print("Type 'quit' to end the session.")
print('-' * 60)

while True:
    learner_input = input('\nYou: ').strip()
    if learner_input.lower() in ['quit', 'exit', 'stop']:
        print('\n' + '=' * 60)
        print(' SESSION SUMMARY')
        print('=' * 60)
        print(f' Topic               : {subject}')
        print(f' Total Turns         : {session.turn}')
        print(f' Final Mastery Score : {session.mastery_score}/100')
        print(f' Final ADI           : {session.adi}/100')
        autonomy = 'High' if session.adi < 40 else 'Medium' if session.adi < 70 else 'Low'
        print(f' Cognitive Autonomy  : {autonomy}')
        print('=' * 60)
        print('\nThank you for learning with LearnIS.')
        break
    if learner_input:
        session.respond(learner_input)

---

## How Gemma 4 Powers LearnIS

| Gemma 4 Capability | LearnIS Usage |
|---|---|
| Native system prompt support | Encodes the full Socratic pedagogical protocol |
| Advanced reasoning (31B) | Maintains coherent multi-turn Socratic dialogue |
| Structured JSON output | Powers real-time ADI and mastery score metrics |
| Open-weight Apache 2.0 | Deployable locally without proprietary API dependency |
| Edge models E2B and E4B | Offline-first operation for low-connectivity environments |

---

## Impact

LearnIS addresses the growing problem of cognitive dependency on AI in education, particularly in sub-Saharan Africa and the Global South where access to qualified tutors is severely limited.

By positioning Gemma 4 as a Socratic partner rather than an answer machine, LearnIS transforms AI from a threat to critical thinking into its greatest ally.

> *LearnIS does not replace human thinking. It trains it.*

---

**Author:** Arnaud Kedagni | [GitHub](https://github.com/kedagniarnaud999-ai/LearnIS) | Gemma 4 Good Hackathon 2026